## Plot geopotential height using Orthograhic Projection and make a movie

In [14]:
!date; hostname; which python

Mon Mar  9 09:26:59 EDT 2026
pp401
/nbhome/ogrp/python/envs/dev/bin/python


In [ ]:
!ls /archive/oar.gfdl.cm5/frerts/FMS2025.03_cm5hires_08062025/cm5_hiresv2_am5f8d6r1_om5_b11_nonbous_piC/gfdl.ncrc6-intel23-prod-openmp/pp/atmos_cmip/ts/monthly/5yr/atmos_cmip.001601-002012.zg.nc

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd


# EDIT: set these
#inputfilepath
DIR = '/archive/oar.gfdl.cm5/frerts/FMS2025.03_cm5hires_08062025/cm5_hiresv2_am5f8d6r1_om5_b11_nonbous_piC/gfdl.ncrc6-intel23-prod-openmp/pp/atmos_cmip/ts/monthly/5yr/'
inputfile_zg = DIR + 'atmos_cmip.001601-002012.zg.nc'
inputfile_ua = DIR + 'atmos_cmip.001601-002012.ua.nc'
inputfile_va = DIR + 'atmos_cmip.001601-002012.va.nc'

#Available pressure levels in file
plevels_hPa = [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 70, 50, 30, 20, 10, 5, 1]
#Desired pressure Level to plot geopotential height 
hPa = 10 
#plev = index where plevels_hPa is hPa
plev = plevels_hPa.index(hPa)
plev

In [ ]:
ds_zg = xr.open_dataset(inputfile_zg, use_cftime=True)
ds_ua = xr.open_dataset(inputfile_ua, use_cftime=True)
ds_va = xr.open_dataset(inputfile_va, use_cftime=True)

In [ ]:
zg_plev = ds_zg.zg[:, plev, :, :]  # adjust dim names if needed
ua_plev = ds_ua.ua[:, plev, :, :]
va_plev = ds_va.va[:, plev, :, :]
zg_plev.shape, ua_plev.shape, va_plev.shape


In [ ]:
ua_plev[0,:,: ].plot()  # example: plot first time level

In [ ]:
z_plot = zg_plev.isel(time=0)
print("Data min:", z_plot.min().values)
print("Data max:", z_plot.max().values)
print("Data shape:", z_plot.shape)
print("NaN count:", z_plot.isnull().sum().values)
print("Lon range:", ds_zg.lon_bnds.min().values, "to", ds_zg.lon_bnds.max().values)
print("Lat range:", ds_zg.lat_bnds.min().values, "to", ds_zg.lat_bnds.max().values)

In [ ]:
lons = ds_zg.lon_bnds[:,0]
lats = ds_zg.lat_bnds[:,0]

# Create 2D meshgrid from 1D coords
lon_2d, lat_2d = np.meshgrid(lons,lats)
lon_2d.shape , lat_2d.shape


In [ ]:
sk=48
skip = (slice(None, None, sk), slice(None, None, sk))
ua = ua_plev.isel(time=10)[skip]
va = va_plev.isel(time=10)[skip]
ua.shape, va.shape, lon_2d[skip].shape, lat_2d[skip].shape, lons[skip[1]].shape, lats[skip[0]].shape, lons[::sk].shape, lats[::sk].shape

In [ ]:
# Set fixed color scale
vmin = zg_plev.min().values
vmax = zg_plev.max().values


In [ ]:
#Test one time step
for t in range(0,1): #z_plev.shape[0],10):
    fig = plt.figure(figsize=(10, 10))
    # Use Orthographic projection for sphere view (centered on North Pole)
    ax = plt.axes(projection=ccrs.Orthographic(central_longitude=0, central_latitude=60))
    # Set global extent
    ax.set_global()

    # Add Map Features
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='grey')
    ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3)
    ax.gridlines(draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
    # Plot the Geopotential Height
    # We use 'coolwarm' or 'viridis' to show the "low" height (cold core) vs "high" height
    z_plot = zg_plev.isel(time=t)
    contour = ax.contourf(
        lon_2d, lat_2d, z_plot.values,
        transform=ccrs.PlateCarree(),
        levels=10,
        cmap="viridis",
        vmin=vmin, vmax=vmax,
    )
    # 7. Add Wind Vectors (Quiver)
    q = ax.quiver(lons[skip[1]].values, lats[skip[0]].values, 
                  ua.values, va.values, 
                  transform=ccrs.PlateCarree(), 
                  color='black', alpha=0.8, scale=500)
        
    plt.colorbar(contour, orientation='horizontal', pad=0.05, label='Geopotential Height (m)')

    date = pd.to_datetime(str(ds_zg.time_bnds[t][0].values))
    plt.title(f"Geopotential Height at {hPa} hPa \nTime {date.strftime('%Y-%m-%d')}", fontsize=15)
    plt.show()

In [ ]:
#Animating over time steps
from matplotlib import animation
from IPython.display import HTML

fig = plt.figure(figsize=(10, 10))
# Use Orthographic projection for sphere view (centered on North Pole)
ax = plt.axes(projection=ccrs.Orthographic(central_longitude=0, central_latitude=60))
# Set global extent
ax.set_global()

# Add Map Features
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='grey')
ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3)
ax.gridlines(draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')

# Initial frame
t0 = 0
z_plot = zg_plev.isel(time=t0)
contour = ax.contourf(
    lon_2d, lat_2d, z_plot.values,
    transform=ccrs.PlateCarree(),
    levels=10,
    cmap="viridis",
    vmin=vmin, vmax=vmax,
)
#Add Wind Vectors (Quiver)
ua = ua_plev.isel(time=t0)[skip]
va = va_plev.isel(time=t0)[skip]

q = ax.quiver(lons[skip[1]].values, lats[skip[0]].values, 
                ua.values, va.values, 
                transform=ccrs.PlateCarree(), 
                color='black', alpha=0.8, scale=500)

cbar = plt.colorbar(contour, orientation='horizontal', pad=0.05, label='Geopotential Height (m)')
title = ax.set_title("")

def update(t):
    for coll in ax.collections:
        coll.remove()
        
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='grey')

    z_plot = zg_plev.isel(time=t)
    ax.contourf(
        lon_2d, lat_2d, z_plot.values,
        transform=ccrs.PlateCarree(),
        levels=10,
        cmap="viridis",
        vmin=vmin, vmax=vmax,
    )
    #Add Wind Vectors (Quiver)
    ua = ua_plev.isel(time=t)[skip]
    va = va_plev.isel(time=t)[skip]

    q = ax.quiver(lons[skip[1]].values, lats[skip[0]].values, 
                    ua.values, va.values, 
                    transform=ccrs.PlateCarree(), 
                    color='black', alpha=0.8, scale=500)

    date = pd.to_datetime(str(ds_zg.time_bnds[t][0].values))
    title.set_text(f"Geopotential Height at {hPa} hPa \nTime {date.strftime('%Y-%m-%d')}")
    return ax.collections + [title]

ani = animation.FuncAnimation(fig, update, frames=60, interval=1000, blit=False)
HTML(ani.to_jshtml())

#took 1m 17.4s

In [ ]:
# Save to MP4
ani.save("geopotheight_"+ str(hPa) +"hPa_OrhtographicProjection.mp4", writer="ffmpeg", dpi=150)
